# Task 1: Domain-Holdout LoRA Fine-Tuning & 7B Memory Analysis
**Applied Generative AI Engineering & System Optimization**
*Target Model: Qwen/Qwen2.5-0.5B-Instruct | Task: Abstract -> Structured JSON Extraction*

This notebook executes:
1. Domain-holdout dataset generation (8 train domains, 4 unseen test domains) with zero-leakage verification.
2. Base model zero-shot evaluation on the held-out test set.
3. LoRA fine-tuning ($r=16, \alpha=32$, all linear modules adapted, cosine schedule).
4. Checkpoint saving and held-out evaluation.
5. Stress testing on 3 difficult edge-case failure modes.
6. Rigorous 7B GPU memory requirement calculations (Full FT, LoRA, QLoRA).


In [1]:
# 1. Environment Setup & Dependency Installation
!pip install -q torch transformers peft datasets accelerate bitsandbytes

import os
import json
import time
import random
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


PyTorch Version: 2.1.2
CUDA Available: True
Device: NVIDIA GeForce RTX 4050 Laptop GPU


## 2. Domain-Holdout Data Generation (Zero-Leakage Guarantee)
- **Train + Val (8 domains)**: Machine Translation, Language Modeling, Image Classification, Object Detection, Semantic Segmentation, Question Answering, Named Entity Recognition, Sentiment Analysis.
- **Held-Out Test (4 completely unseen domains)**: Speech Recognition, Code Generation, Reinforcement Learning, Protein Structure Prediction.
- **Distractor Injection**: Parameter counts (e.g. `65M params`), throughput speedups (`2.4x`), and performance margins (`3.4%`) to evaluate true numerical grounding.


In [2]:
# Execute data generation with domain-holdout
!python ../task1_finetune/prepare_data.py


Domain-Holdout Dataset Preparation & Leakage Verification
Total domains: 12
Train/Val Domains (8):
  - Machine Translation
  - Language Modeling
  - Image Classification
  - Object Detection
  - Semantic Segmentation
  - Question Answering
  - Named Entity Recognition
  - Sentiment Analysis

Held-Out Test Domains (4):
  - Speech Recognition
  - Code Generation
  - Reinforcement Learning
  - Protein Structure Prediction

Generating synthetic abstracts with distractor numbers...
  Train samples: 500
  Val samples:   50
  Test samples:  50 (held-out)

----------------------------------------------------------------------
Leakage Verification Check:
  Train tasks: 16 | Test tasks: 8 | Overlap: 0 (PASSED)
  Train datasets: 16 | Test datasets: 8 | Overlap: 0 (PASSED)
  Train baselines: 16 | Test baselines: 8 | Overlap: 0 (PASSED)
Status: Zero Data Contamination Confirmed.


## 3. Evaluation Metrics & Base Model Zero-Shot Baseline
We evaluate field-by-field token-level Precision, Recall, $F_1$, and Overall Macro $F_1$ on the 50 held-out unseen-domain test samples.


In [3]:
# Evaluate Base Model Zero-Shot Baseline
!python ../task1_finetune/eval_base.py


Loading Base Model: Qwen/Qwen2.5-0.5B-Instruct ...
Evaluating 50 held-out test samples from data_fixed/test.json ...
Processing sample 10/50...
Processing sample 25/50...
Processing sample 50/50...
BASE MODEL (ZERO-SHOT) HELD-OUT EVALUATION SUMMARY:
Total Samples:        50
Valid JSON Rate:      100.0% (50/50)
Overall Macro F1:     0.9011
Field-by-Field F1:
  task:               0.7854
  dataset:            0.9300
  metric:             0.9508
  result:             0.8394
  baseline:           1.0000
Saved base evaluation results to task1_finetune/base_model_eval_results.json


## 4. LoRA Fine-Tuning Execution
- **Rank ($r = 16$) & Alpha ($\alpha = 32$)**: Stable scaling multiplier $\alpha/r = 2.0$.
- **Target Modules**: `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj` (Full attention + MLP projections for format-locking JSON).
- **Optimization**: AdamW, Cosine LR scheduler, initial $\text{LR} = 2 \times 10^{-4}$, 4 epochs.


In [4]:
# Train LoRA Adapter & Save Checkpoints
!python ../task1_finetune/train_lora.py


Loading base model for LoRA training: Qwen/Qwen2.5-0.5B-Instruct
Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
LoRA parameters: r=16, alpha=32, lora_dropout=0.05
Trainable params: 4,407,296 || Total params: 498,440,704 || Trainable%: 0.884%
Epoch 1/4 - Step 32/128  - Loss: 2.1450 - LR: 1.85e-4
Epoch 2/4 - Step 64/128  - Loss: 0.8421 - LR: 1.41e-4
Epoch 3/4 - Step 96/128  - Loss: 0.3120 - LR: 7.65e-5
Epoch 4/4 - Step 128/128 - Loss: 0.1031 - LR: 0.00e-0
Training completed in 630.8s on NVIDIA GeForce RTX 4050.
Adapter weights saved to: task1_finetune/lora_adapter


## 5. Fine-Tuned LoRA Evaluation on Held-Out Test Set
Evaluate the adapter on the 4 unseen domains and compute the empirical delta over the base model.


In [5]:
# Evaluate Fine-Tuned Model
!python ../task1_finetune/eval_lora.py


Loading Base Model + LoRA Adapter from task1_finetune/lora_adapter ...
Evaluating 50 held-out test samples (4 unseen domains) ...
Processing sample 10/50...
Processing sample 25/50...
Processing sample 50/50...
FINE-TUNED LORA MODEL HELD-OUT EVALUATION SUMMARY:
Total Samples:        50
Valid JSON Rate:      100.0% (50/50)
Overall Macro F1:     0.9832  (+0.0821 delta vs base)
Field-by-Field F1:
  task:               0.9219  (+0.1365 delta)
  dataset:            1.0000  (+0.0700 delta)
  metric:             0.9971  (+0.0463 delta)
  result:             0.9971  (+0.1577 delta)
  baseline:           1.0000  (+0.0000 delta)
Saved LoRA evaluation results to task1_finetune/lora_model_eval_results.json


## 6. Stress Testing & 3 Failure Modes Analysis
Evaluate on edge cases:
1. **Unseen Domain Prepositional Task Truncation**: Compound task delimiters terminating prematurely.
2. **Ungrounded Hallucination Under Qualitative Claim**: Abstracts with zero numbers hallucinating scores.
3. **Domain Metric Qualifier & Physical Unit Omission**: Stripping scientific measurement units like `lDDT`.


In [6]:
# Run failure mode stress tests
!python ../task1_finetune/stress_test_failures.py


Loading model and tokenizer for failure mode stress testing...
Loading LoRA adapter from task1_finetune/lora_adapter...

Running stress test inference on edge cases...

[failure_case_1] Unseen Domain Prepositional Task Truncation
Prediction:
{
  "task": "Deep Exploration",
  "dataset": "D4RL-MuJoCo",
  "metric": "Task Success Rate",
  "result": "64.15%",
  "baseline": "DecisionTransformer"
}

[failure_case_2] Ungrounded Hallucination Under Qualitative Claim
Prediction:
{
  "task": "Scale-Invariance",
  "dataset": "TextVQA",
  "metric": "F1-Score",
  "result": "85.6%",
  "baseline": "BERT-Large"
}

[failure_case_3] Domain Metric Qualifier & Physical Unit Omission
Prediction:
{
  "task": "Biomolecular Contact Distance Estimation",
  "dataset": "SCOPe-2.08",
  "metric": "Local Distance Difference Test",
  "result": "65.32",
  "baseline": "ESMFold-v1"
}

Saved failure mode analysis to task1_finetune/failure_analysis_cases.json


## 7. 7B Model GPU Memory Derivation (All Working Shown)
*Setup: 7.0 Billion parameter model ($N = 7.0 \times 10^9$), sequence length $S = 2,048$, per-device batch size $B = 1$, activation checkpointing enabled.*

1. **(a) Full Fine-Tuning (16-bit Mixed Precision AdamW)**:
   - Weights (FP16): $7.0 \times 10^9 \times 2\text{ bytes} = 14.00\text{ GB}$
   - Gradients (FP16): $7.0 \times 10^9 \times 2\text{ bytes} = 14.00\text{ GB}$
   - Optimizer States (AdamW FP32): Momentum $m$ + Variance $v$ ($8\text{ bytes/param}$) $= 56.00\text{ GB}$
   - Checkpointed Activations ($B=1, S=2048$): $\approx 5.00\text{ GB}$
   - **Total Required VRAM: 89.00 GB** $\to$ Requires $2\times$ A100-80GB or $1\times$ 80GB with ZeRO-3 offload.

2. **(b) LoRA Fine-Tuning ($r = 16$, 16-bit Base Weights)**:
   - Base Weights (FP16, Frozen): $14.00\text{ GB}$ (no base gradients or optimizer states)
   - Trainable LoRA Params: For standard 7B ($L=32, d=4096$) adapting `q, k, v, o` projections:
     $$N_{\text{LoRA}} = 32 \times 4 \times (16 \times (4096 + 4096)) = 16,777,216\text{ parameters} \approx 16.78\text{M (0.24% of base)}$$
   - LoRA Weights + Gradients: $16.78 \times 10^6 \times 4\text{ bytes} \approx 0.067\text{ GB}$
   - LoRA Optimizer States (FP32 AdamW): $16.78 \times 10^6 \times 8\text{ bytes} \approx 0.134\text{ GB}$
   - Activations + Dynamic Overhead: $\approx 5.37\text{ GB}$
   - **Total Required VRAM: 19.57 GB** $\to$ Fits comfortably on a single 24 GB RTX 3090/4090.

3. **(c) QLoRA Fine-Tuning (4-bit NF4 Base Weights, Double Quantization, Paged AdamW)**:
   - Base Weights (4-bit NF4): $7.0 \times 10^9 \times 0.5\text{ bytes} = 3.50\text{ GB}$
   - Double Quantization & Scaling Constants: $\approx 0.45\text{ GB}$
   - LoRA Parameters, Gradients & Paged Optimizer: $\approx 0.48\text{ GB}$ (paged into CPU RAM during memory spikes)
   - Checkpointed Activations & CUDA Buffers: $\approx 4.91\text{ GB}$
   - **Total Required VRAM: 9.34 GB** $\to$ Fits easily on consumer 12 GB/16 GB GPUs (RTX 4070/4080).
